## Podcast Listening Time Prediction

This project predicts how many minutes a listener will spend on a podcast episode, using episode metadata such as the podcast, genre, episode length, host and guest popularity, publication day and time, number of ads, and episode sentiment.

## Approach
1. Load the data and explore the target and missing values
2. Fill missing values and encode categorical features
3. Train a baseline regression model and evaluate RMSE on a validation split
4. Generate predictions and create the submission file
5. Submit to Kaggle and record the score

In [1]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c playground-series-s5e4
!unzip -oq playground-series-s5e4.zip

Paste your Kaggle API token and press Enter: ··········
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 4.2 MB/s eta 0:00:00
100% 22.3M/22.3M [00:00<00:00, 129MB/s] 



In [2]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
train.shape, test.shape

((750000, 12), (250000, 11))

In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 12 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           750000 non-null  int64  
 1   Podcast_Name                 750000 non-null  object 
 2   Episode_Title                750000 non-null  object 
 3   Episode_Length_minutes       662907 non-null  float64
 4   Genre                        750000 non-null  object 
 5   Host_Popularity_percentage   750000 non-null  float64
 6   Publication_Day              750000 non-null  object 
 7   Publication_Time             750000 non-null  object 
 8   Guest_Popularity_percentage  603970 non-null  float64
 9   Number_of_Ads                749999 non-null  float64
 10  Episode_Sentiment            750000 non-null  object 
 11  Listening_Time_minutes       750000 non-null  float64
dtypes: float64(5), int64(1), object(6)
memory usage: 68.7+ MB


In [5]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 11 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           250000 non-null  int64  
 1   Podcast_Name                 250000 non-null  object 
 2   Episode_Title                250000 non-null  object 
 3   Episode_Length_minutes       221264 non-null  float64
 4   Genre                        250000 non-null  object 
 5   Host_Popularity_percentage   250000 non-null  float64
 6   Publication_Day              250000 non-null  object 
 7   Publication_Time             250000 non-null  object 
 8   Guest_Popularity_percentage  201168 non-null  float64
 9   Number_of_Ads                250000 non-null  float64
 10  Episode_Sentiment            250000 non-null  object 
dtypes: float64(4), int64(1), object(6)
memory usage: 21.0+ MB


In [6]:
train.sample(1)

,id,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
642322,642322,Business Insights,Episode 37,111.74,Business,37.72,Monday,Evening,17.7,0.0,Positive,90.73375


In [7]:
train.dtypes

,0
id,int64
Podcast_Name,object
Episode_Title,object
Episode_Length_minutes,float64
Genre,object
Host_Popularity_percentage,float64
Publication_Day,object
Publication_Time,object
Guest_Popularity_percentage,float64
Number_of_Ads,float64


In [8]:
train.isnull().sum()[lambda s: s > 0]

,0
Episode_Length_minutes,87093
Guest_Popularity_percentage,146030
Number_of_Ads,1


In [9]:
train.select_dtypes(include="object").nunique()

,0
Podcast_Name,48
Episode_Title,100
Genre,10
Publication_Day,7
Publication_Time,4
Episode_Sentiment,3


In [10]:
test_ids = test["id"]
train = train.drop(columns=["id", "Episode_Title"])
test = test.drop(columns=["id", "Episode_Title"])

In [11]:
train["Episode_Length_minutes"] = train["Episode_Length_minutes"].fillna(train["Episode_Length_minutes"].median())

In [12]:
train["Guest_Popularity_percentage"] = train["Guest_Popularity_percentage"].fillna(0)

In [13]:
train["Number_of_Ads"] = train["Number_of_Ads"].fillna(train["Number_of_Ads"].median())

In [14]:
test["Episode_Length_minutes"] = test["Episode_Length_minutes"].fillna(test["Episode_Length_minutes"].median())

In [15]:
test["Guest_Popularity_percentage"] = test["Guest_Popularity_percentage"].fillna(0)

In [16]:
test["Number_of_Ads"] = test["Number_of_Ads"].fillna(test["Number_of_Ads"].median())

In [17]:
train.isnull().sum().sum()

np.int64(0)

In [18]:
test.isnull().sum().sum()

np.int64(0)

In [19]:
train = pd.get_dummies(train, columns=["Podcast_Name", "Genre", "Publication_Day", "Publication_Time", "Episode_Sentiment"])

In [20]:
test = pd.get_dummies(test, columns=["Podcast_Name", "Genre", "Publication_Day", "Publication_Time", "Episode_Sentiment"])

In [21]:
train, test = train.align(test, join="left", axis=1, fill_value=0)

In [22]:
train.shape, test.shape

((750000, 77), (250000, 77))

In [24]:
x = train.drop(columns=["Listening_Time_minutes"])
y = train["Listening_Time_minutes"]

In [25]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=60, max_depth=10, min_samples_leaf=20, random_state=42, n_jobs=-1)
model.fit(x_train, y_train)

pred = model.predict(x_val)
rmse = mean_squared_error(y_val, pred) ** 0.5
print("Validation RMSE:", round(rmse, 3))

Validation RMSE: 13.086


In [26]:
model.fit(x, y)

x_test = test.drop(columns=["Listening_Time_minutes"])
pred_test = model.predict(x_test)

pd.DataFrame({"id": test_ids, "Listening_Time_minutes": pred_test}).to_csv("submission.csv", index=False)

In [27]:
!kaggle competitions submit -c playground-series-s5e4 -f submission.csv -m "RandomForest baseline"

100% 6.01M/6.01M [00:00<00:00, 12.8MB/s]
99 submissions remaining today.
Successfully submitted to Predict Podcast Listening Time

In [28]:
import pickle, os
pickle.dump(model, open("podcast_model.pkl", "wb"))
print("size MB:", round(os.path.getsize("podcast_model.pkl") / 1e6, 1))

size MB: 8.1
